# MangaWhisperer — Pilot: narrate one Berserk volume

**Before running:**
1. Upload the project folder to Google Drive at `MyDrive/MangaWhisperer/narracao_imersiva_manga`
   (skip `.venv/` and `workspace/` — everything else, including `data_berserk_samples/`, comes along).
2. `Runtime → Change runtime type → T4 GPU` (this notebook is preset, but double-check).
3. **Plan A (Claude API):** click the key icon (Secrets) in the left sidebar, add a secret named
   `ANTHROPIC_API_KEY`, and enable notebook access. **Plan B (no key):** do nothing — the pipeline
   automatically uses the free local Qwen-VL model instead.

Run the cells top to bottom. If Colab disconnects mid-run, just reconnect and re-run —
every stage checkpoints to disk, so completed work (including paid Claude calls) is never repeated.

In [ ]:
# 1. Confirm we actually have a GPU
!nvidia-smi

In [ ]:
# 2. Mount Drive and copy the project to fast local disk
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/MangaWhisperer'
PROJECT_SRC = f'{DRIVE_ROOT}/narracao_imersiva_manga'
PROJECT = '/content/mangawhisperer'

!rsync -a --exclude='.venv' --exclude='workspace' --exclude='.pytest_cache' "{PROJECT_SRC}/" "{PROJECT}/"
!ls "{PROJECT}"

In [ ]:
# 3. Install the package with all engine extras (~5 min; heavy: coqui-tts, transformers, easyocr)
%pip install -q -e "/content/mangawhisperer[dev,engines,vlm,vlm-local,tts]"
print('Install done. If Colab suggests restarting the session, do it, then re-run from cell 2.')

In [ ]:
# 4. Sanity check: run the full offline test suite (should be ~50 tests, all green, <1 min)
!cd /content/mangawhisperer && python -m pytest tests -q

In [ ]:
# 5. Credentials + licenses
import os
os.environ['COQUI_TOS_AGREED'] = '1'  # XTTSv2 CPML license (non-commercial accessibility use)

try:
    from google.colab import userdata
    key = userdata.get('ANTHROPIC_API_KEY')
    if key:
        os.environ['ANTHROPIC_API_KEY'] = key
except Exception:
    pass

if os.environ.get('ANTHROPIC_API_KEY'):
    print('Plan A: Claude API key loaded — scriptwriter will be claude-opus-4-8')
else:
    print('Plan B: no API key — scriptwriter will be local Qwen2.5-VL (free, slower, weaker PT-BR)')

In [ ]:
# 6. SMOKE TEST first: 10 pages end-to-end (~10-25 min incl. model downloads).
#    Listen to the result before committing to the full volume.
PDF = f'{DRIVE_ROOT}/narracao_imersiva_manga/data_berserk_samples/Berserk-20260706T192902Z-3-002/Berserk/BERSERK VOL.01.pdf'

!cd /content/mangawhisperer && python scripts/run_pilot.py \
    --pdf "{PDF}" --workspace /content/workspace --start 8 --pages 10 --vlm auto

In [ ]:
# 7. Listen + inspect the diarized script
import json, pathlib
from IPython.display import Audio, display

ws = pathlib.Path('/content/workspace/berserk_vol_01')
final = next((ws / 'final').glob('*.wav'))
print(f'{final} ({final.stat().st_size / 1e6:.1f} MB)')

panels = json.loads((ws / 'script' / 'panels.json').read_text(encoding='utf-8'))
for panel in panels[:8]:
    for block in panel['blocks']:
        tag = 'fala' if block['is_speech'] else 'ação'
        print(f"[p{panel['page_number']:>3}] {block['speaker_id']:>14} ({tag}): {block['text']}")

display(Audio(str(final)))

In [ ]:
# 8. FULL VOLUME (several hours on a T4 — safe to leave running; it resumes if disconnected).
#    Note: this reuses the same workspace, so the 10 smoke pages' script is NOT reused
#    (different page range) — pass --fresh to be explicit about starting over.
!cd /content/mangawhisperer && python scripts/run_pilot.py \
    --pdf "{PDF}" --workspace /content/workspace_full --vlm auto

In [ ]:
# 9. Export results back to Drive (WAV + MP3 + the script checkpoints)
OUT = f'{DRIVE_ROOT}/output/berserk_vol_01'
!mkdir -p "{OUT}"

SRC = '/content/workspace_full/berserk_vol_01'
!cp "{SRC}/final"/*.wav "{OUT}/" 2>/dev/null || cp /content/workspace/berserk_vol_01/final/*.wav "{OUT}/"
!cp "{SRC}/script/panels.json" "{OUT}/" 2>/dev/null || cp /content/workspace/berserk_vol_01/script/panels.json "{OUT}/"

# WAV -> MP3 (much smaller for listening on a phone)
import glob
for wav in glob.glob(f'{OUT}/*.wav'):
    !ffmpeg -y -loglevel error -i "{wav}" -b:a 96k "{wav[:-4]}.mp3"
!ls -lh "{OUT}"

## Troubleshooting

- **Session disconnected mid-run** — reconnect, re-run cells 2-5, then re-run the run cell.
  Checkpoints under the workspace mean completed stages (incl. Claude calls) are skipped.
  Caveat: `/content` is wiped on a *full* runtime recycle — for very long runs, point
  `--workspace` at a Drive path instead (slower I/O, but survives everything).
- **Qwen out-of-memory on T4** — add `--qwen-model Qwen/Qwen2.5-VL-3B-Instruct` to the run command.
- **XTTS license prompt** — cell 5 sets `COQUI_TOS_AGREED=1` (CPML: non-commercial use).
- **Costs** — the smoke run (10 pages ≈ 50 panels) costs roughly $1-2 on claude-opus-4-8, ~$0.20 on
  haiku; the full volume ≈ $30 opus / ~$5 haiku. GPU time is covered by Colab free/Pro.
- **Try a cheaper Claude model** — edit `build_vlm()` in `scripts/run_pilot.py`:
  `ClaudeVisionLanguageEngine(model='claude-haiku-4-5')`, re-run the smoke, compare the scripts.